# 7 dpa to 10 dpa CNS Morphogenesis

Infer 7-to-10 dpa CNS cell mappings, reconstruct the SparseVFC morphofield, integrate trajectories, and calculate morphogenetic features without Dynamo.

This curated notebook targets the current Dynamo-free Spateo API. Edit the path/configuration cells for a new system before execution.


In [ ]:
import pyvista as pv

pv.global_theme.transparent_background = True
import numpy as np

import spateo as st


In [ ]:
cpo = [
    (1828.9380323399394, -749.256660402299, 3711.3578588227624),
    (1117.1464767456055, -10.815200805664062, 32.063289642333984),
    (-0.20648289903881356, 0.9508440558374369, 0.23078213510395512),
]


## Load and validate data


In [ ]:
stage1_adata = st.read_h5ad("/DATA/User/gaomohan/figures2_\u8865\u5145/data/7dpa_tdr_aligned.h5ad")
stage2_adata = st.read_h5ad("/DATA/User/gaomohan/figures2_\u8865\u5145/data/10dpa_tdr_aligned.h5ad")
stage1_adata, stage2_adata


In [ ]:
stage1_cns = stage1_adata[stage1_adata.obs["anno"].astype(str) == "CNS"].copy()
stage2_cns = stage2_adata[stage2_adata.obs["anno"].astype(str) == "CNS"].copy()
stage1_cns, stage2_cns


In [ ]:
# Stage 1: 7 dpa
stage1_nonzero = np.asarray(stage1_cns.layers["counts_X"].sum(axis=0)).ravel() > 0
# Stage 2: 10 dpa
stage2_nonzero = np.asarray(stage2_cns.layers["counts_X"].sum(axis=0)).ravel() > 0

stage2_detected = set(stage2_cns.var_names[stage2_nonzero])
common_genes = [gene for gene in stage1_cns.var_names[stage1_nonzero] if gene in stage2_detected]
if not common_genes:
    raise ValueError("No shared nonzero genes were found between the two CNS stages.")

stage1_cns = stage1_cns[:, common_genes].copy()
stage2_cns = stage2_cns[:, common_genes].copy()
for adata in (stage1_cns, stage2_cns):
    st.pp.normalize_total(
        adata,
        layer="counts_X",
        out_layer="norm_X",
        target_sum=None,
        size_factor_key="Size_Factor",
        inplace=True,
    )
    st.pp.log1p_layer(
        adata,
        layer="norm_X",
        out_layer="log1p_X",
        set_X=True,
        inplace=True,
    )

print(f"Shared nonzero genes: {len(common_genes):,}")


## Construct the point-cloud model


In [ ]:
cpo = [
    (746.583638250998, 1426.5293241132986, 4650.867243041787),
    (944.0, 1655.0, 75.0),
    (-0.9837490807936163, 0.17637001290509832, -0.033635763490226143),
]

stage1_aligned_pc, _ = st.tdr.construct_pc(
    adata=stage1_adata.copy(),
    spatial_key="3d_align_spatial_nonrigid",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)

stage2_aligned_pc, _ = st.tdr.construct_pc(
    adata=stage2_adata.copy(),
    spatial_key="3d_align_spatial_nonrigid",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)

raw_pair = st.tdr.collect_models([stage1_aligned_pc.copy(), stage2_aligned_pc.copy()])
st.pl.three_d_plot(
    model=raw_pair,
    key="groups",
    colormap=["#DC143C", "#0000FF"],
    model_style="points",
    model_size=2,
    opacity=[0.6, 0.6],
    show_legend=False,
    show_axes=True,
    jupyter="static",
    window_size=(400, 400),
    cpo=cpo
    # filename = f"./figures/10dpa_14dpa_nonrigid_aligned.pdf",
)


## Infer cross-stage cell directions


In [ ]:
st.tdr.cell_directions(
    adataA=stage1_cns,
    adataB=stage2_cns,
    layer="log1p_X",
    numItermaxEmd=500000,
    spatial_key="3d_align_spatial",
    key_added="cells_mapping",
    alpha=0.0001,
    device="0",
    inplace=True,
)


## Reconstruct the surface mesh


In [ ]:
stage1_aligned_pc, _ = st.tdr.construct_pc(
    adata=stage1_cns,
    spatial_key="3d_align_spatial",
    groupby="anno",
    key_added="tissue",
    colormap={"CNS": "#DC143C"},
)

stage1_aligned_mesh, _, _ = st.tdr.construct_surface(
    pc=stage1_aligned_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.3},
    smooth=5000,
    scale_factor=1.02,
)

stage2_aligned_pc, _ = st.tdr.construct_pc(
    adata=stage2_cns,
    spatial_key="3d_align_spatial",
    groupby="anno",
    key_added="tissue",
    colormap={"CNS": "#DC143C"},
)

stage2_aligned_mesh, _, _ = st.tdr.construct_surface(
    pc=stage2_aligned_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.5},
    smooth=5000,
    scale_factor=1.02,
)


In [ ]:
align_lines, _ = st.tdr.construct_align_lines(
    model1_points=stage1_cns.obsm["3d_align_spatial"].copy(),
    model2_points=stage1_cns.obsm["X_cells_mapping"].copy() + np.asarray([0, 0, -300]),
    key_added="check_align",
    label="align_lines",
    color="gainsboro",
)

stage1_aligned_pc_v = stage1_aligned_pc.copy()
stage2_aligned_pc_v = stage2_aligned_pc.copy()
stage2_aligned_pc_v.points[:, 2] = stage2_aligned_pc_v.points[:, 2] - 300


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([align_lines, stage1_aligned_pc_v, stage2_aligned_pc_v]),
    colormap=["gainsboro", "red", "blue"],
    opacity=[0.05, 1.0, 1.0],
    model_style=["wireframe", "points", "points"],
    model_size=[2, 3, 3],
    show_legend=False,
    off_screen=False,
    show_axes=True,
    jupyter="static",
    # background="black",
    window_size=(512, 512),
    cpo="iso",
    # filename = f"./figures/10dpa_14dpa_mapcell.pdf",
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([stage1_aligned_mesh, stage1_aligned_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=False,
    jupyter="static",
    window_size=(400, 400),
    cpo="iso",
    # filename = f"./figures/10dpa_mesh.pdf",
)


## Reconstruct the developmental vector field


In [ ]:
st.tdr.morphofield_sparsevfc(
    adata=stage1_cns,
    spatial_key="3d_align_spatial",
    V_key="V_cells_mapping",
    key_added="VecFld_morpho",
    NX=np.asarray(stage1_aligned_mesh.points),
    inplace=True,
)
stage1_aligned_pc.point_data["vectors"] = stage1_cns.uns["VecFld_morpho"]["V"]
stage1_aligned_mesh.point_data["vectors"] = stage1_cns.uns["VecFld_morpho"]["grid_V"]


### Data-adaptive arrow scaling

Arrow scaling is a display choice. The current Spateo-native SparseVFC preserves spatial-displacement units, so historical factors tuned to Dynamo-compressed vectors must not be reused. The helper below targets arrows at 1.5% of the model bounding-box diagonal at the 95th vector-magnitude percentile.


In [ ]:
def vector_display_factor(model, vector_key="vectors", target_fraction=0.015, quantile=0.95):
    # Scale arrows relative to the model size without changing the vector field.
    vectors = np.asarray(model.point_data[vector_key], dtype=float)
    magnitudes = np.linalg.norm(vectors, axis=1)
    positive = magnitudes[np.isfinite(magnitudes) & (magnitudes > 0)]
    if positive.size == 0:
        raise ValueError(f"No finite nonzero vectors found in {vector_key!r}.")
    model_diagonal = np.linalg.norm(np.ptp(np.asarray(model.points), axis=0))
    return target_fraction * model_diagonal / np.quantile(positive, quantile)

point_arrow_factor = vector_display_factor(stage1_aligned_pc)
mesh_arrow_factor = vector_display_factor(stage1_aligned_mesh)
print(f"Point-cloud arrow factor: {point_arrow_factor:.4g}")
print(f"Mesh arrow factor: {mesh_arrow_factor:.4g}")


In [ ]:
vector_arrows1, _ = st.tdr.construct_field(
    model=stage1_aligned_pc,
    vf_key="vectors",
    arrows_scale_key="vectors",
    n_sampling=None,
    factor=point_arrow_factor,
    key_added="v_arrows",
    label=stage1_aligned_pc.point_data["vectors"][:, 2].flatten(),
    color="rainbow",
)

st.pl.three_d_plot(
    model=st.tdr.collect_models([stage1_aligned_pc, vector_arrows1]),
    key=["tissue", "v_arrows"],
    colormap=["gainsboro", "rainbow"],
    opacity=[0.5, 1],
    model_style=["points", "surface"],
    model_size=3,
    show_legend=False,
    legend_kwargs=dict(title="", fmt="%.2e", legend_loc=(0.87, 0.3), label_font_size=20),
    off_screen=False,
    show_axes=True,
    jupyter="static",
    # background="black",
    window_size=(1024, 1024),
    cpo=None,
    # filename = f"./figures/10dpa_vector_pc.pdf",
)


In [ ]:
vector_arrows2, _ = st.tdr.construct_field(
    model=stage1_aligned_mesh,
    vf_key="vectors",
    arrows_scale_key="vectors",
    factor=mesh_arrow_factor,
    # n_sampling=2500,
    # sampling_method = "random",
    key_added="v_arrows",
    label=stage1_aligned_mesh.point_data["vectors"][:, 2].flatten(),
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([stage1_aligned_mesh, vector_arrows2]),
    key=["tissue", "v_arrows"],
    opacity=[0.2, 1],
    colormap=["gainsboro", "rainbow"],
    model_style="surface",
    show_legend=False,
    show_axes=True,
    off_screen=False,
    jupyter="static",
    background="black",
    window_size=(1024, 1024),
    cpo=None,
    # filename = f"./figures/10dpa_vector_mesh.pdf"
)


## Predict developmental trajectories


In [ ]:
st.tdr.morphopath(
    adata=stage1_cns,
    layer="log1p_X",
    vf_key="VecFld_morpho",
    key_added="fate_morpho",
    t_end=5e4,
    interpolation_num=50,
    cores=20,
)
stage1_cns


In [ ]:
trajectory_model, _ = st.tdr.construct_trajectory(
    adata=stage1_cns,
    fate_key="fate_morpho",
    sampling_method="trn",
    label=stage1_cns.uns["VecFld_morpho"]["V"][:, 2].flatten(),
    trajectory_color="rainbow",
    tip_color="rainbow",
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([stage1_aligned_pc, trajectory_model]),
    key=["tissue", "trajectory"],
    opacity=[0.2, 0.5],
    model_style=["points", "wireframe"],
    model_size=[5, 3],
    colormap=["gainsboro", "rainbow"],
    show_legend=False,
    show_axes=True,
    off_screen=False,
    jupyter="static",
    # background="black",
    window_size=(1024, 1024),
    cpo=None,
    # filename = f"./figures/10dpa_cell_developmental_trajectory_pc.pdf",
)


## Reconstruct trajectories for per-cell feature visualization


In [ ]:
trajectory_model, _ = st.tdr.construct_trajectory(
    adata=stage1_cns,
    fate_key="fate_morpho",
    key_added="obs_index",
    # n_sampling=500,
    # sampling_method="trn",
    label=np.asarray(stage1_cns.obs.index),
)


In [ ]:
glm_dict = {}


In [ ]:
cpo = [
    (2285.6730347755797, 1667.246650908813, 2722.3111036682994),
    (639.8601271078041, 988.4251013251522, 160.2615182700938),
    (-0.5354950817756835, -0.6652659273091227, 0.5202559594618664),
]


## Calculate morphogenetic features


In [ ]:
key = "velocity"
st.tdr.morphofield_velocity(
    adata=stage1_cns,
    vf_key="VecFld_morpho",
    key_added=key,
)
stage1_cns.obsm["velocity"]


In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning

warnings.filterwarnings(
    "ignore",
    message=".*Negative binomial dispersion parameter alpha not set.*",
    category=ValueWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*Gene expression matrix must be normalized by the size factor.*",
)


In [ ]:
key = "acceleration"
glm_key = f"glm_degs_{key}"
st.tdr.morphofield_acceleration(
    adata=stage1_cns,
    vf_key="VecFld_morpho",
    key_added=key,
)


## Associate gene expression with morphology


In [ ]:
st.tl.glm_degs(
    adata=stage1_cns,
    layer=None,
    fullModelFormulaStr="~cr({key}, df=3)",
    key_added=glm_key,
    qval_threshold=0.001,
    llf_threshold=-450,
)
glm_data = stage1_cns.uns[glm_key]["glm_result"]
glm_dict[key] = glm_data
glm_data


In [ ]:
glm_data.to_csv("glm_data_acceleration_7dpa_10dpa.csv", index=True)


In [ ]:
st.pl.glm_fit(
    adata=stage1_cns,
    genes=glm_data.index.tolist(),
    ncols=4,
    feature_x=key,
    feature_y="expression",
    glm_key=glm_key,
    save_show_or_return="show",
)


## Interpolate gene expression in 3D


In [ ]:
# Candidate genes detected by the curl and/or acceleration GLMs.
for gn in [
    "SMESG000026791.1",
    "SMESG000019950.1",
    "SMESG000039758.1",
    "SMESG000034720.1",
    "SMESG000013290.1",
    "SMESG000046043.1",
]:
    interpolated_gp_adata = st.tdr.gp_interpolation(
        source_adata=stage1_cns.copy(),
        spatial_key="3d_align_spatial",
        keys=gn,
        target_points=np.asarray(stage1_aligned_pc.points),
        device="0",
        training_iter=100,
    )
    interpolated_gp_pc, _ = st.tdr.construct_pc(
        adata=interpolated_gp_adata.copy(), spatial_key="3d_align_spatial", groupby=gn, key_added=gn
    )
    _gn = str(gn).replace(":", "_") if ":" in gn else gn

    st.pl.three_d_plot(
        model=interpolated_gp_pc,
        key=gn,
        model_style="points",
        model_size=7,
        opacity=0.2,
        colormap="Reds",
        show_legend=False,
        jupyter="static",
        off_screen=False,
        cpo=None,
        window_size=(512, 512),
        text=gn,
        # filename = f"./figures/10dpa_{_gn}_interpolated_gp_pc.pdf"
    )


In [ ]:
# Genes detected by both the curl and acceleration analyses.
gene_list = [
    "SMESG000019950.1",  # SPP-9 curl acceleration
    "SMESG000039758.1",  # 1020HH-1 curl acceleration
    "SMESG000013290.1",  # pyrokinin prohormone-like-1 curl acceleration
    #'SMESG000034720.1',#npp-18 only curl
    #'SMESG000026791.1'#estrella only curl
]


In [ ]:
st.pl.glm_fit(
    adata=stage1_cns,
    genes=gene_list,
    ncols=2,
    feature_x=key,
    feature_y="expression",
    glm_key=glm_key,
    save_show_or_return="show",
)


In [ ]:
st.pl.acceleration(
    adata=stage1_cns,
    model=st.tdr.collect_models([stage1_aligned_pc, trajectory_model]),
    acceleration_key=key,
    colormap="default_cmap",
    jupyter="static",
    model_style=["points", "wireframe"],
    model_size=[5, 2],
    cpo=cpo,
    show_legend=False,
    window_size=(1024, 1024),
    background="white",
    legend_kwargs=dict(title="", fmt="%.2e", legend_loc=(0.87, 0.3), label_font_size=20),
    # filename = f"./figures/10dpa_acceleration_trajectory_pc.pdf",
)


In [ ]:
key = "curl"
glm_key = f"glm_degs_{key}"
st.tdr.morphofield_curl(
    adata=stage1_cns,
    vf_key="VecFld_morpho",
    key_added=key,
)


In [ ]:
st.tl.glm_degs(
    adata=stage1_cns,
    layer=None,
    fullModelFormulaStr="~cr({key}, df=3)",
    key_added=glm_key,
    qval_threshold=0.01,
    llf_threshold=-490,
)
glm_data = stage1_cns.uns[glm_key]["glm_result"]


In [ ]:
glm_dict[key] = glm_data
glm_data


In [ ]:
glm_data.to_csv("glm_data_curl_7dpa_10dpa.csv", index=True)


In [ ]:
# Genes detected by both the curl and acceleration analyses.
gene_list = [
    "SMESG000019950.1",  # SPP-9 curl acceleration
    "SMESG000039758.1",  # 1020HH-1 curl acceleration
    "SMESG000013290.1",  # pyrokinin prohormone-like-1 curl acceleration
    "SMESG000034720.1",  # npp-18 only curl
    "SMESG000026791.1",  # estrella only curl
]


In [ ]:
genes = [
    "SMESG000069088.1",
    "SMESG000080359.1",
    "SMESG000038885.1",
    "SMESG000068721.1",
    "SMESG000033458.1",
    "SMESG000006806.1",
    "SMESG000076695.1",
    "SMESG000042475.1",
    "SMESG000069610.1",
    "SMESG000043039.1",
    "SMESG000016568.1",
]

for gene in genes:
    print(gene, gene in glm_data.index)


In [ ]:
st.pl.curl(
    adata=stage1_cns,
    # model=trajectory_model,
    model=st.tdr.collect_models([stage1_aligned_pc, trajectory_model]),
    curl_key=key,
    colormap="default_cmap",
    jupyter="static",
    model_style=["points", "wireframe"],
    model_size=[5, 2],
    show_legend=False,
    cpo=cpo,
    window_size=(1024, 1024),
    background="white",
    legend_kwargs=dict(title="", fmt="%.2e", legend_loc=(0.87, 0.3), label_font_size=20),
    # filename = f"./figures/10dpa_curl_trajectory_pc.pdf",
)


In [ ]:
key = "divergence"
glm_key = f"glm_degs_{key}"
st.tdr.morphofield_divergence(
    adata=stage1_cns,
    vf_key="VecFld_morpho",
    key_added=key,
)


In [ ]:
st.pl.divergence(
    adata=stage1_cns,
    # model=trajectory_model,
    model=st.tdr.collect_models([stage1_aligned_pc, trajectory_model]),
    divergence_key=key,
    colormap="default_cmap",
    jupyter="static",
    model_style=["points", "wireframe"],
    model_size=[5, 2],
    cpo=None,
    window_size=(1024, 1024),
    background="white",
    legend_kwargs=dict(title="", fmt="%.2e", legend_loc=(0.87, 0.3), label_font_size=20),
    # filename = f"./figures/10dpa_divergence_trajectory_pc.pdf",
)


In [ ]:
key = "torsion"
glm_key = f"glm_degs_{key}"
st.tdr.morphofield_torsion(
    adata=stage1_cns,
    vf_key="VecFld_morpho",
    key_added=key,
)


In [ ]:
st.pl.torsion(
    adata=stage1_cns,
    # model=trajectory_model,
    model=st.tdr.collect_models([stage1_aligned_pc, trajectory_model]),
    torsion_key=key,
    colormap="default_cmap",
    jupyter="static",
    model_style=["points", "wireframe"],
    model_size=[5, 2],
    cpo=cpo,
    window_size=(1024, 1024),
    background="white",
    show_legend=False,
    legend_kwargs=dict(title="", fmt="%.2e", legend_loc=(0.87, 0.3), label_font_size=20),
    # filename = f"./figures/10dpa_torsion_trajectory_pc.pdf",
)


In [ ]:
key = "curvature"
glm_key = f"glm_degs_{key}"
st.tdr.morphofield_curvature(
    adata=stage1_cns,
    vf_key="VecFld_morpho",
    key_added=key,
)


In [ ]:
st.pl.curvature(
    adata=stage1_cns,
    # model=trajectory_model,
    model=st.tdr.collect_models([stage1_aligned_pc, trajectory_model]),
    curvature_key=key,
    colormap="default_cmap",
    jupyter="static",
    model_style=["points", "wireframe"],
    model_size=[5, 2],
    cpo=None,
    window_size=(1024, 1024),
    background="white",
    legend_kwargs=dict(title="", fmt="%.2e", legend_loc=(0.87, 0.3), label_font_size=20),
    # filename = f"./figures/10dpa_curvature_trajectory_pc.pdf",
)
